<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent; border-radius: 14px; padding: 18px 22px; margin: 12px 0;
  font-size: 36px; font-weight: 600; color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25); background-clip: padding-box; position: relative;
">
  <div style="position: absolute; inset: 0; padding: 4px; border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: linear-gradient(#fff 0 0) content-box, linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor; mask-composite: exclude; pointer-events: none;"></div>
  <b>00. Introduction to Gaussian Processes</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [What is a Gaussian Process?](#-part-i-what-is-a-gaussian-process)**
    - 1.1 [From Functions to Distributions over Functions](#from-functions)
    - 1.2 [The Kernel (Covariance Function)](#the-kernel)
- **2. [Setup & Imports](#-part-ii-setup--imports)**
- **3. [GP Prior: Sampling Random Functions](#-part-iii-gp-prior)**
    - 3.1 [Visualizing the GP Prior](#visualizing-gp-prior)
    - 3.2 [Effect of Kernel Parameters](#effect-of-kernel-params)
- **4. [GP Regression with TFP](#-part-iv-gp-regression)**
    - 4.1 [Fitting a GP to Data](#fitting-a-gp)
    - 4.2 [Posterior Predictions with Uncertainty](#posterior-predictions)
- **5. [Kernel Hyperparameter Optimization](#-part-v-kernel-optimization)**
- **6. [GP vs Neural Networks](#-part-vi-gp-vs-nn)**
- **7. [Key Takeaways](#-part-vii-key-takeaways)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. What is a Gaussian Process?</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">1.1. From Functions to Distributions over Functions</span>

A **Gaussian Process (GP)** is a collection of random variables, any finite number of which have a joint Gaussian distribution.

$$f \sim \mathcal{GP}(m(x), k(x, x'))$$

Where:
- $m(x)$ is the **mean function** (often assumed to be zero)
- $k(x, x')$ is the **kernel (covariance function)** — encodes our assumptions about function smoothness

**Key Insight**: A GP defines a **distribution over functions**. Instead of parameterizing a function explicitly (like a neural network), we define a prior over the entire function space.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">1.2. The Kernel (Covariance Function)</span>

| **Kernel** | **Formula** | **Properties** | **TFP Class** |
|-----------|------------|----------------|---------------|
| RBF (Squared Exponential) | $k(x,x') = \sigma^2 \exp\left(-\frac{\|x-x'\|^2}{2\ell^2}\right)$ | Infinitely smooth | `ExponentiatedQuadratic` |
| Matérn 3/2 | $k(x,x') = \sigma^2(1+\frac{\sqrt{3}r}{\ell})\exp(-\frac{\sqrt{3}r}{\ell})$ | Once differentiable | `MaternThreeHalves` |
| Matérn 5/2 | $k(x,x') = \sigma^2(1+\frac{\sqrt{5}r}{\ell}+\frac{5r^2}{3\ell^2})\exp(-\frac{\sqrt{5}r}{\ell})$ | Twice differentiable | `MaternFiveHalves` |
| Linear | $k(x,x') = \sigma^2 x \cdot x'$ | Models linear trends | `Linear` |
| Periodic | $k(x,x') = \sigma^2 \exp\left(-\frac{2\sin^2(\pi|x-x'|/p)}{\ell^2}\right)$ | Periodic patterns | `ExpSinSquared` |

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Imports</span>

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt

tfd = tfp.distributions
tfk = tfp.math.psd_kernels  # Positive semidefinite kernels

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow: {tf.__version__}")
print(f"TFP: {tfp.__version__}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. GP Prior: Sampling Random Functions</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.1. Visualizing the GP Prior</span>

In [ ]:
# Define a GP prior with RBF kernel
amplitude = 1.0
length_scale = 0.5

kernel = tfk.ExponentiatedQuadratic(
    amplitude=amplitude,
    length_scale=length_scale
)

# Create GP prior
index_points = np.linspace(-5, 5, 200).reshape(-1, 1).astype(np.float32)

gp_prior = tfd.GaussianProcess(
    kernel=kernel,
    index_points=index_points,
    observation_noise_variance=1e-6  # Small jitter for numerical stability
)

# Sample functions from the prior
fig, ax = plt.subplots(figsize=(14, 7))

n_samples = 5
prior_samples = gp_prior.sample(n_samples)

colors = ['#ef4444', '#3b82f6', '#22c55e', '#f59e0b', '#8b5cf6']
for i in range(n_samples):
    ax.plot(index_points[:, 0], prior_samples[i].numpy(), 
            color=colors[i], linewidth=2, alpha=0.8, label=f'Sample {i+1}')

# Show 2-sigma confidence band
mean = gp_prior.mean().numpy()
std = gp_prior.stddev().numpy()
ax.fill_between(index_points[:, 0], mean - 2*std, mean + 2*std,
                alpha=0.15, color='gray', label='±2σ prior')

ax.set_xlabel('x', fontsize=14)
ax.set_ylabel('f(x)', fontsize=14)
ax.set_title(f'GP Prior: Random Functions (RBF kernel, ℓ={length_scale}, σ={amplitude})',
             fontsize=16, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.2. Effect of Kernel Parameters</span>

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Vary length scale
length_scales = [0.2, 0.5, 2.0]
for i, ls in enumerate(length_scales):
    k = tfk.ExponentiatedQuadratic(amplitude=1.0, length_scale=ls)
    gp = tfd.GaussianProcess(kernel=k, index_points=index_points, 
                             observation_noise_variance=1e-6)
    samples = gp.sample(3)
    for j in range(3):
        axes[0, i].plot(index_points[:, 0], samples[j].numpy(), 
                        linewidth=2, alpha=0.7)
    axes[0, i].set_title(f'RBF (ℓ={ls})', fontsize=14, fontweight='bold')
    axes[0, i].set_ylim(-3, 3)

# Different kernel types
kernels = [
    ('Matérn 3/2', tfk.MaternThreeHalves(amplitude=1.0, length_scale=0.5)),
    ('Matérn 5/2', tfk.MaternFiveHalves(amplitude=1.0, length_scale=0.5)),
    ('RBF', tfk.ExponentiatedQuadratic(amplitude=1.0, length_scale=0.5)),
]
for i, (name, k) in enumerate(kernels):
    gp = tfd.GaussianProcess(kernel=k, index_points=index_points,
                             observation_noise_variance=1e-6)
    samples = gp.sample(3)
    for j in range(3):
        axes[1, i].plot(index_points[:, 0], samples[j].numpy(),
                        linewidth=2, alpha=0.7)
    axes[1, i].set_title(f'{name} Kernel', fontsize=14, fontweight='bold')
    axes[1, i].set_ylim(-3, 3)

axes[0, 0].set_ylabel('Varying Length Scale', fontsize=13)
axes[1, 0].set_ylabel('Varying Kernel Type', fontsize=13)

plt.suptitle('Effect of Kernel Choice on GP Samples', fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. GP Regression with TFP</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">4.1. Fitting a GP to Data</span>

In [ ]:
# Generate noisy training data
np.random.seed(42)
n_train = 20
X_train = np.sort(np.random.uniform(-4, 4, n_train)).astype(np.float32).reshape(-1, 1)
y_train = (np.sin(X_train[:, 0]) + 0.2 * np.random.randn(n_train)).astype(np.float32)

# GP Regression using the posterior (GaussianProcessRegressionModel)
kernel = tfk.ExponentiatedQuadratic(
    amplitude=tf.constant(1.0),
    length_scale=tf.constant(0.5)
)

# Create the GP regression model (posterior conditioned on training data)
X_test = np.linspace(-5, 5, 200).astype(np.float32).reshape(-1, 1)

gprm = tfd.GaussianProcessRegressionModel(
    kernel=kernel,
    index_points=X_test,
    observation_index_points=X_train,
    observations=y_train,
    observation_noise_variance=0.04,  # σ_noise²
    predictive_noise_variance=0.0     # Predict the function, not noisy observations
)

# Get posterior mean and standard deviation
post_mean = gprm.mean().numpy()
post_std = gprm.stddev().numpy()

print(f"Posterior predictive shape: {post_mean.shape}")
print(f"Mean range: [{post_mean.min():.2f}, {post_mean.max():.2f}]")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">4.2. Posterior Predictions with Uncertainty</span>

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

# Sample posterior functions
n_posterior_samples = 5
post_samples = gprm.sample(n_posterior_samples)
for i in range(n_posterior_samples):
    ax.plot(X_test[:, 0], post_samples[i].numpy(), color='#8b5cf6', 
            alpha=0.2, linewidth=1)

# Uncertainty bands
ax.fill_between(X_test[:, 0], post_mean - 2*post_std, post_mean + 2*post_std,
                alpha=0.15, color='#3b82f6', label='±2σ (95% CI)')
ax.fill_between(X_test[:, 0], post_mean - post_std, post_mean + post_std,
                alpha=0.25, color='#3b82f6', label='±1σ (68% CI)')

# Posterior mean
ax.plot(X_test[:, 0], post_mean, color='#3b82f6', linewidth=2.5, label='Posterior mean')

# True function
ax.plot(X_test[:, 0], np.sin(X_test[:, 0]), 'r--', linewidth=2, label='True: sin(x)')

# Training data
ax.scatter(X_train[:, 0], y_train, c='#1e293b', s=80, zorder=5,
           edgecolors='white', linewidth=2, label='Training data')

ax.set_xlabel('x', fontsize=14)
ax.set_ylabel('f(x)', fontsize=14)
ax.set_title('Gaussian Process Regression\nUncertainty grows away from observed data',
             fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.set_xlim(-5, 5)
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">5. Kernel Hyperparameter Optimization</span>

We optimize kernel hyperparameters by maximizing the **marginal likelihood** (evidence).

In [ ]:
# Trainable kernel hyperparameters
log_amplitude = tf.Variable(0.0, name='log_amplitude')
log_length_scale = tf.Variable(0.0, name='log_length_scale')
log_noise = tf.Variable(-2.0, name='log_noise_variance')

def build_gp():
    kernel = tfk.ExponentiatedQuadratic(
        amplitude=tf.exp(log_amplitude),
        length_scale=tf.exp(log_length_scale)
    )
    return tfd.GaussianProcess(
        kernel=kernel,
        index_points=X_train,
        observation_noise_variance=tf.exp(log_noise)
    )

# Optimize marginal likelihood
optimizer = tf.keras.optimizers.Adam(learning_rate=0.05)
losses = []

for step in range(300):
    with tf.GradientTape() as tape:
        gp = build_gp()
        nll = -gp.log_prob(y_train)
    
    grads = tape.gradient(nll, [log_amplitude, log_length_scale, log_noise])
    optimizer.apply_gradients(zip(grads, [log_amplitude, log_length_scale, log_noise]))
    losses.append(nll.numpy())
    
    if step % 50 == 0:
        print(f"Step {step}: NLL={nll:.4f}, amplitude={tf.exp(log_amplitude):.3f}, "
              f"length_scale={tf.exp(log_length_scale):.3f}, "
              f"noise_var={tf.exp(log_noise):.4f}")

print(f"\nOptimized: amplitude={tf.exp(log_amplitude):.3f}, "
      f"length_scale={tf.exp(log_length_scale):.3f}, "
      f"noise_var={tf.exp(log_noise):.4f}")

In [ ]:
# GP regression with optimized hyperparameters
opt_kernel = tfk.ExponentiatedQuadratic(
    amplitude=tf.exp(log_amplitude),
    length_scale=tf.exp(log_length_scale)
)

opt_gprm = tfd.GaussianProcessRegressionModel(
    kernel=opt_kernel,
    index_points=X_test,
    observation_index_points=X_train,
    observations=y_train,
    observation_noise_variance=tf.exp(log_noise),
    predictive_noise_variance=0.0
)

opt_mean = opt_gprm.mean().numpy()
opt_std = opt_gprm.stddev().numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Training curve
axes[0].plot(losses, color='#3b82f6', linewidth=2)
axes[0].set_xlabel('Step', fontsize=13)
axes[0].set_ylabel('Negative Log Marginal Likelihood', fontsize=13)
axes[0].set_title('Kernel Hyperparameter Optimization', fontsize=15, fontweight='bold')

# Optimized GP
axes[1].fill_between(X_test[:, 0], opt_mean - 2*opt_std, opt_mean + 2*opt_std,
                     alpha=0.2, color='#3b82f6', label='±2σ')
axes[1].plot(X_test[:, 0], opt_mean, color='#3b82f6', linewidth=2.5, label='Posterior mean')
axes[1].plot(X_test[:, 0], np.sin(X_test[:, 0]), 'r--', linewidth=2, label='True function')
axes[1].scatter(X_train[:, 0], y_train, c='#1e293b', s=60, zorder=5,
                edgecolors='white', linewidth=2, label='Data')
axes[1].set_xlabel('x', fontsize=13)
axes[1].set_ylabel('f(x)', fontsize=13)
axes[1].set_title('GP with Optimized Hyperparameters', fontsize=15, fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">6. GP vs Neural Networks</span>

| **Property** | **Gaussian Processes** | **Neural Networks** |
|-------------|----------------------|--------------------|
| Uncertainty | ✅ Built-in, calibrated | ❌ Requires special methods |
| Data efficiency | ✅ Great with small data | ❌ Need large datasets |
| Scalability | ❌ O(N³) — limited to ~10K points | ✅ Scales to millions |
| Flexibility | ⚠️ Limited by kernel choice | ✅ Universal approximators |
| Interpretability | ✅ Kernel is interpretable | ❌ Black box |
| Hyperparameters | Few (kernel params) | Many (architecture, lr, etc.) |

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">7. Key Takeaways</span>

| **Concept** | **Key Point** |
|------------|---------------|
| GP = distribution over functions | Defined by mean function + kernel |
| Kernel | Encodes smoothness, periodicity, and other prior beliefs |
| GP regression | Closed-form posterior: exact Bayesian inference! |
| Marginal likelihood | Use it to optimize kernel hyperparameters |
| Uncertainty | Grows in regions far from training data |
| TFP classes | `GaussianProcess`, `GaussianProcessRegressionModel` |

### Next Steps
- **Notebook 01**: GP Classification and kernel engineering
- **Notebook 02**: Sparse and scalable GPs for large datasets